# IAM RAG Project - Embedding Generation

This notebook generates three types of embedding vectors for the IAM document collection:
- **Dense Embedding**: Uses Voyage AI to generate dense vectors (for semantic search)
- **Sparse Embedding**: Uses the SPLADE model to generate sparse vectors (for precise search)
- **BM25 Index**: Builds a keyword retrieval index using the BM25 algorithm

The output is stored in the corresponding subdirectories under the `indexes/` directory.

## 1. Dense Embedding - Voyage Code-3

In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount('/content/drive') #force_remount=True
print("Drive mounted successfully!")


Mounted at /content/drive
Drive mounted successfully!


In [ ]:
import os

CANDIDATES = [
    '/content/drive/MyDrive/iam_rag_project',
    '/content/drive/MyDrive/iam_rag_project (1)',
]

PROJECT_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find iam_rag_project folder")

print("Using project root:", PROJECT_ROOT)

Using project root: /content/drive/MyDrive/iam_rag_project


In [ ]:
CHUNK_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed', 'chunking')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'indexes')

In [ ]:
!pip install -q voyageai rank_bm25 transformers torch tqdm faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 50.7 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
os.environ['VOYAGE_API_KEY'] = userdata.get('VOYAGE_API_KEY')

import voyageai

client = voyageai.Client()

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/iam_rag_project'
CHUNK_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed', 'chunking')
index_DIR = os.path.join(PROJECT_ROOT, 'indexes', 'dense')

In [ ]:
import json
from tqdm import tqdm


def embed_chunks_with_voyage(chunks_path, output_path, batch_size=128):
    """
    Generates dense embeddings for chunks using Voyage Code-3

    Args:
        chunks_path: Path to the input chunks JSON file
        output_path: Path to the output JSON file with embeddings
        batch_size: Number of chunks to process in each batch (default 128)
    """
    # Load raw chunks
    with open(chunks_path, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    print(f"Loading {len(chunks)} chunks from {chunks_path}...")
    embeddings = []

    # Call API in batches to generate embeddings
    for i in tqdm(range(0, len(chunks), batch_size)):
        batch_chunks = chunks[i:i + batch_size]
        batch_texts = [chunk['text'] for chunk in batch_chunks]

        response = client.embed(
            texts=batch_texts,
            model="voyage-code-3",
            input_type="document"
        )
        embeddings.extend(response.embeddings)

    # Build result: combine text, metadata, and embedding
    result = []
    for chunk, embedding in zip(chunks, embeddings):
        result.append({
            "text": chunk["text"],
            "metadata": chunk.get("metadata", {}),
            "embedding": embedding
        })

    # Save to output file
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False)

    print(f"✓ Saved {len(result)} embeddings to {output_path}\n")

embed_chunks_with_voyage(
    chunks_path=os.path.join(CHUNK_DIR, 'action_chunks.json'),
    output_path=os.path.join(OUTPUT_DIR, 'dense', 'action_chunks_embedded.json')
)

embed_chunks_with_voyage(
    chunks_path=os.path.join(CHUNK_DIR, 'policy_chunks.json'),
    output_path=os.path.join(OUTPUT_DIR, 'dense','policy_chunks_embedded.json')
)

Loading 4505 chunks from /content/drive/MyDrive/iam_rag_project/data/processed/chunking/action_chunks.json...


100%|██████████| 36/36 [00:59<00:00,  1.65s/it]


✓ Saved 4505 embeddings to /content/drive/MyDrive/iam_rag_project/indexes/dense/action_chunks_embedded.json

Loading 500 chunks from /content/drive/MyDrive/iam_rag_project/data/processed/chunking/policy_chunks.json...


100%|██████████| 4/4 [00:09<00:00,  2.33s/it]


✓ Saved 500 embeddings to /content/drive/MyDrive/iam_rag_project/indexes/dense/policy_chunks_embedded.json



In [ ]:
import json
import numpy as np
import faiss
import os
import pickle

def combine_and_build_dense_index(
    action_embedded_path,
    policy_embedded_path,
    output_dir
):
    """
    Combine action and policy embedded chunks into
    one unified FAISS index for dense retrieval.
    """
    # Load both embedded files
    with open(action_embedded_path) as f:
        action_chunks = json.load(f)
    with open(policy_embedded_path) as f:
        policy_chunks = json.load(f)

    # Combine into one list
    # ORDER MATTERS — position in list = position in FAISS index
    combined = action_chunks + policy_chunks

    print(f"Combining dense chunks:")
    print(f"  Action chunks:  {len(action_chunks)}")
    print(f"  Policy chunks:  {len(policy_chunks)}")
    print(f"  Combined total: {len(combined)}")

    # Save combined chunks for metadata lookup at retrieval time
    combined_chunks_path = os.path.join(
        output_dir,'indexes', 'dense', 'combined_chunks.json'
    )
    with open(combined_chunks_path, 'w') as f:
        json.dump(combined, f)

    # Extract embeddings into numpy matrix
    vectors = np.array(
        [c['embedding'] for c in combined],
        dtype='float32'
    )
    print(f"  Vector matrix shape: {vectors.shape}")

    # Normalize for cosine similarity
    faiss.normalize_L2(vectors)

    # Build FAISS index
    dim   = vectors.shape[1]          # 1024 for voyage-code-3
    index = faiss.IndexFlatIP(dim)    # inner product after normalization
    index.add(vectors)

    # Save FAISS index
    faiss_index_path = os.path.join(
        output_dir, 'indexes','dense', 'combined_faiss.index'
    )
    faiss.write_index(index, faiss_index_path)

    print(f"  ✓ FAISS index saved: {faiss_index_path}")
    print(f"  ✓ Combined chunks saved: {combined_chunks_path}")

    return index, combined


dense_index, dense_chunks = combine_and_build_dense_index(
    action_embedded_path=os.path.join(
        PROJECT_ROOT, 'indexes','dense', 'action_chunks_embedded.json'
    ),
    policy_embedded_path=os.path.join(
        PROJECT_ROOT, 'indexes','dense', 'policy_chunks_embedded.json'
    ),
    output_dir=PROJECT_ROOT
)

Combining dense chunks:
  Action chunks:  4505
  Policy chunks:  500
  Combined total: 5005
  Vector matrix shape: (5005, 1024)
  ✓ FAISS index saved: /content/drive/MyDrive/iam_rag_project/indexes/dense/combined_faiss.index
  ✓ Combined chunks saved: /content/drive/MyDrive/iam_rag_project/indexes/dense/combined_chunks.json


## 2. Sparse Embedding - SPLADE

In [ ]:
import json
import torch
from tqdm import tqdm
from transformers import AutoModelForMaskedLM, AutoTokenizer

# initialize the SPLADE model and the tokenizer
model_id = 'naver/splade-v3'

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForMaskedLM.from_pretrained(model_id, token=hf_token)

# use GPU or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"Using device: {device}")


def generate_splade_embeddings(chunks_path, output_path, batch_size=16):
    """
    Generates sparse vectors for chunks using the SPLADE model

    Args:
        chunks_path: Path to the input chunks JSON file
        output_path: Path to the output JSON file with sparse embeddings
        batch_size: Number of chunks to process in each batch (default 16)
    """
    # Load raw chunks
    with open(chunks_path, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    print(f"Loading {len(chunks)} chunks from {chunks_path}...")
    sparse_embeddings = []

    # Process chunks in batches
    for i in tqdm(range(0, len(chunks), batch_size)):
        batch_chunks = chunks[i:i + batch_size]
        batch_texts = [chunk['text'] for chunk in batch_chunks]

        # Tokenize text
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        # Generate SPLADE sparse vectors
        with torch.no_grad():
            logits = model(**inputs).logits
            # SPLADE core: extract sparse weights via log1p(ReLU(logits))
            logits_relu = torch.log1p(torch.relu(logits))
            token_max, _ = torch.max(
                logits_relu * inputs.attention_mask.unsqueeze(-1),
                dim=1
            )

        # Convert tensor to sparse dictionary format (only store non-zero weights)
        for j in range(len(batch_texts)):
            # Extract token IDs with non-zero weights
            nonzero_indices = torch.nonzero(token_max[j]).squeeze(-1).cpu().tolist()
            nonzero_weights = token_max[j][nonzero_indices].cpu().tolist()

            # Convert to {token_id: weight} dictionary
            sparse_dict = {
                str(idx): round(weight, 4)
                for idx, weight in zip(nonzero_indices, nonzero_weights)
            }
            sparse_embeddings.append(sparse_dict)

    # Build result: combine text, metadata, and sparse embedding
    result = []
    for chunk, sparse_vec in zip(chunks, sparse_embeddings):
        result.append({
            "text": chunk["text"],
            "metadata": chunk.get("metadata", {}),
            "sparse_embedding": sparse_vec
        })

    # Save to output file
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False)

    print(f"✓ Saved {len(result)} SPLADE embeddings to {output_path}\n")

generate_splade_embeddings(
    chunks_path=os.path.join(CHUNK_DIR, 'action_chunks.json'),
    output_path=os.path.join(OUTPUT_DIR, 'sparse', 'action_chunks_splade.json')
)

generate_splade_embeddings(
    chunks_path=os.path.join(CHUNK_DIR, 'policy_chunks.json'),
    output_path=os.path.join(OUTPUT_DIR, 'sparse', 'policy_chunks_splade.json')
)

config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Error during conversion: AttributeError("'str' object has no attribute 'decode'")


Loading 4505 chunks from /content/drive/MyDrive/iam_rag_project/data/processed/chunking/action_chunks.json...


  0%|          | 0/282 [00:00<?, ?it/s]Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The repo does not appear to have a file named 

✓ Saved 4505 SPLADE embeddings to /content/drive/MyDrive/iam_rag_project/indexes/sparse/action_chunks_splade.json



"\ngenerate_splade_embeddings(\n    chunks_path=os.path.join(CHUNK_DIR, 'policy_chunks.json'),\n    output_path=os.path.join(OUTPUT_DIR, 'sparse', 'policy_chunks_splade.json')\n)\n"

In [ ]:
def combine_splade_chunks(
    action_splade_path,
    policy_splade_path,
    output_dir
):
    """
    Combine action and policy SPLADE chunks into
    one unified list for sparse retrieval.
    No index needed — dot product computed at query time.
    """
    with open(action_splade_path) as f:
        action_chunks = json.load(f)
    with open(policy_splade_path) as f:
        policy_chunks = json.load(f)

    combined = action_chunks + policy_chunks

    print(f"Combining SPLADE chunks:")
    print(f"  Action chunks:  {len(action_chunks)}")
    print(f"  Policy chunks:  {len(policy_chunks)}")
    print(f"  Combined total: {len(combined)}")

    # Save combined
    combined_path = os.path.join(
        output_dir, 'indexes','sparse', 'combined_chunks_splade.json'
    )
    with open(combined_path, 'w') as f:
        json.dump(combined, f)

    print(f"  ✓ Combined SPLADE chunks saved: {combined_path}")

    return combined


# Run
splade_chunks = combine_splade_chunks(
    action_splade_path=os.path.join(
        PROJECT_ROOT, 'indexes','sparse', 'action_chunks_splade.json'
    ),
    policy_splade_path=os.path.join(
        PROJECT_ROOT, 'indexes','sparse', 'policy_chunks_splade.json'
    ),
    output_dir=PROJECT_ROOT
)

Combining SPLADE chunks:
  Action chunks:  4505
  Policy chunks:  500
  Combined total: 5005
  ✓ Combined SPLADE chunks saved: /content/drive/MyDrive/iam_rag_project/indexes/sparse/combined_chunks_splade.json


## 3. Keyword Index - BM25

In [ ]:
import json
import os
import pickle
from tqdm import tqdm
from rank_bm25 import BM25Okapi


def tokenize(text):
    """
    A simple tokenization function (English + code)

    For Chinese text, it is recommended to use jieba:
        import jieba
        return list(jieba.cut(text))
    """
    return text.lower().split()


def build_bm25_index(chunks_path, index_output_path, chunks_output_path):
    """
    Builds a BM25 retrieval index

    Args:
        chunks_path: Path to the input chunks JSON file
        index_output_path: Save path for the BM25 index model (.pkl format)
        chunks_output_path: Save path for the chunks data copy (used to query original text during retrieval)
    """
    # Load raw chunks
    with open(chunks_path, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    print(f"Loading {len(chunks)} chunks from {chunks_path}...")

    # Tokenize all chunks
    tokenized_corpus = []
    for chunk in tqdm(chunks, desc="Tokenizing"):
        tokenized_corpus.append(tokenize(chunk['text']))

    # Build BM25 index
    print("Building BM25 Okapi index...")
    bm25 = BM25Okapi(tokenized_corpus)

    # Ensure output directory exists
    os.makedirs(os.path.dirname(index_output_path), exist_ok=True)
    os.makedirs(os.path.dirname(chunks_output_path), exist_ok=True)

    # Save BM25 index model (Python object)
    with open(index_output_path, 'wb') as f:
        pickle.dump(bm25, f)

    # Save chunks copy (for retrieving original text and metadata)
    with open(chunks_output_path, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, ensure_ascii=False)

    print(f"✓ Saved BM25 index to {index_output_path}")
    print(f"✓ Saved chunks mapping to {chunks_output_path}\n")


# Build BM25 index for Action Chunks
build_bm25_index(
    chunks_path=os.path.join(CHUNK_DIR, 'action_chunks.json'),
    index_output_path=os.path.join(OUTPUT_DIR,'bm25', 'action_bm25.pkl'),
    chunks_output_path=os.path.join(OUTPUT_DIR,'bm25', 'action_chunks_mapped.json')
)


# Build BM25 index for Policy Chunks
build_bm25_index(
    chunks_path=os.path.join(CHUNK_DIR, 'policy_chunks.json'),
    index_output_path=os.path.join(OUTPUT_DIR,'bm25', 'policy_bm25.pkl'),
    chunks_output_path=os.path.join(OUTPUT_DIR,'bm25', 'policy_chunks_mapped.json')
)


Loading 4505 chunks from /content/drive/MyDrive/iam_rag_project/data/processed/chunking/action_chunks.json...


Tokenizing: 100%|██████████| 4505/4505 [00:00<00:00, 130417.02it/s]

Building BM25 Okapi index...


✓ Saved BM25 index to /content/drive/MyDrive/iam_rag_project/indexes/bm25/action_bm25.pkl
✓ Saved chunks mapping to /content/drive/MyDrive/iam_rag_project/indexes/bm25/action_chunks_mapped.json



"\n\n# 构建 Policy Chunks 的 BM25 索引\nbuild_bm25_index(\n    chunks_path=os.path.join(CHUNK_DIR, 'policy_chunks.json'),\n    index_output_path=os.path.join(OUTPUT_DIR,'bm25', 'policy_bm25.pkl'),\n    chunks_output_path=os.path.join(OUTPUT_DIR,'bm25', 'policy_chunks_mapped.json')\n)\n"

In [ ]:
def combine_and_build_bm25_index(
    action_chunks_path,
    policy_chunks_path,
    output_dir
):
    """
    Combine action and policy chunks and rebuild
    BM25 index on the combined corpus.

    IMPORTANT: BM25 index must be built on combined corpus
    from scratch — cannot merge two separate BM25 indexes.
    """
    import re
    from rank_bm25 import BM25Okapi
    from tqdm import tqdm

    def tokenize(text):
        text = text.lower()
        text = re.sub(r'[^\w\s:\-]', ' ', text)
        return text.split()

    # Load both chunk files
    with open(action_chunks_path) as f:
        action_chunks = json.load(f)
    with open(policy_chunks_path) as f:
        policy_chunks = json.load(f)

    # Combine — same order used for index and chunks list
    combined = action_chunks + policy_chunks

    print(f"Combining BM25 chunks:")
    print(f"  Action chunks:  {len(action_chunks)}")
    print(f"  Policy chunks:  {len(policy_chunks)}")
    print(f"  Combined total: {len(combined)}")

    # Tokenize all chunks
    print("  Tokenizing...")
    tokenized_corpus = [
        tokenize(chunk['text'])
        for chunk in tqdm(combined)
    ]

    # Build BM25 index on combined corpus
    print("  Building BM25 index...")
    bm25 = BM25Okapi(tokenized_corpus)

    # Save BM25 index
    index_path = os.path.join(
        output_dir,'indexes', 'bm25', 'combined_bm25.pkl'
    )
    with open(index_path, 'wb') as f:
        pickle.dump(bm25, f)

    # Save combined chunks for metadata lookup
    chunks_path = os.path.join(
        output_dir,'indexes', 'bm25', 'combined_chunks_mapped.json'
    )
    with open(chunks_path, 'w') as f:
        json.dump(combined, f)

    print(f"  ✓ BM25 index saved: {index_path}")
    print(f"  ✓ Combined chunks saved: {chunks_path}")

    return bm25, combined


# Run
bm25_index, bm25_chunks = combine_and_build_bm25_index(
    action_chunks_path=os.path.join(
        PROJECT_ROOT, 'indexes','bm25', 'action_chunks_mapped.json'
    ),
    policy_chunks_path=os.path.join(
        PROJECT_ROOT, 'indexes','bm25', 'policy_chunks_mapped.json'
    ),
    output_dir=PROJECT_ROOT
)

Combining BM25 chunks:
  Action chunks:  4505
  Policy chunks:  500
  Combined total: 5005
  Tokenizing...


100%|██████████| 5005/5005 [00:00<00:00, 77112.22it/s]

  Building BM25 index...


  ✓ BM25 index saved: /content/drive/MyDrive/iam_rag_project/indexes/bm25/combined_bm25.pkl
  ✓ Combined chunks saved: /content/drive/MyDrive/iam_rag_project/indexes/bm25/combined_chunks_mapped.json


In [ ]:
import json
import numpy as np
import faiss
import pickle
import os
from collections import Counter

print("=" * 60)
print("COMPLETE EMBEDDING VERIFICATION")
print("=" * 60)

# ── Define expected counts ─────────────────────────────────────────────
EXPECTED_ACTION_CHUNKS  = 4505
EXPECTED_POLICY_CHUNKS  = 500
EXPECTED_TOTAL_CHUNKS   = 5005
EXPECTED_DENSE_DIM      = 1024
EXPECTED_VOCAB_SIZE     = 30522

all_checks_passed = True

def check(condition, message, details=''):
    global all_checks_passed
    status = "✓" if condition else "✗"
    print(f"  {status} {message}")
    if details:
        print(f"    {details}")
    if not condition:
        all_checks_passed = False
    return condition


# ══════════════════════════════════════════════════════════════════════
# SECTION 1: DENSE EMBEDDING (VOYAGE)
# ══════════════════════════════════════════════════════════════════════
print()
print("── 1. Dense Embedding (Voyage) ──────────────────────────────")

# Load files
with open(os.path.join(OUTPUT_DIR, 'dense', 'action_chunks_embedded.json')) as f:
    dense_actions = json.load(f)
with open(os.path.join(OUTPUT_DIR, 'dense', 'policy_chunks_embedded.json')) as f:
    dense_policies = json.load(f)
with open(os.path.join(OUTPUT_DIR, 'dense', 'combined_chunks.json')) as f:
    dense_combined = json.load(f)
dense_index = faiss.read_index(
    os.path.join(OUTPUT_DIR, 'dense', 'combined_faiss.index')
)

# Count checks
check(len(dense_actions)  == EXPECTED_ACTION_CHUNKS,
      f"Action chunks count",
      f"expected={EXPECTED_ACTION_CHUNKS} actual={len(dense_actions)}")
check(len(dense_policies) == EXPECTED_POLICY_CHUNKS,
      f"Policy chunks count",
      f"expected={EXPECTED_POLICY_CHUNKS} actual={len(dense_policies)}")
check(len(dense_combined) == EXPECTED_TOTAL_CHUNKS,
      f"Combined chunks count",
      f"expected={EXPECTED_TOTAL_CHUNKS} actual={len(dense_combined)}")
check(dense_index.ntotal  == EXPECTED_TOTAL_CHUNKS,
      f"FAISS index count",
      f"expected={EXPECTED_TOTAL_CHUNKS} actual={dense_index.ntotal}")

# Dimension checks
check(dense_index.d == EXPECTED_DENSE_DIM,
      f"FAISS dimension",
      f"expected={EXPECTED_DENSE_DIM} actual={dense_index.d}")
check(len(dense_actions[0]['embedding']) == EXPECTED_DENSE_DIM,
      f"Action embedding dimension",
      f"expected={EXPECTED_DENSE_DIM} actual={len(dense_actions[0]['embedding'])}")

# Missing embeddings
missing_action = sum(1 for c in dense_actions  if not c.get('embedding'))
missing_policy = sum(1 for c in dense_policies if not c.get('embedding'))
check(missing_action == 0, f"No missing action embeddings",
      f"missing={missing_action}")
check(missing_policy == 0, f"No missing policy embeddings",
      f"missing={missing_policy}")

# Type distribution in combined
type_counts = Counter(c['metadata']['type'] for c in dense_combined)
check(type_counts['iam_action'] == EXPECTED_ACTION_CHUNKS,
      f"Combined action type count",
      f"expected={EXPECTED_ACTION_CHUNKS} actual={type_counts['iam_action']}")
check(type_counts['iam_policy'] == EXPECTED_POLICY_CHUNKS,
      f"Combined policy type count",
      f"expected={EXPECTED_POLICY_CHUNKS} actual={type_counts['iam_policy']}")

# Order check
check(dense_combined[0]['metadata']['type']  == 'iam_action',
      f"Combined starts with action")
check(dense_combined[-1]['metadata']['type'] == 'iam_policy',
      f"Combined ends with policy")

# Normalization check on sample vectors
sample_indices = [0, 100, 500, 2386, 2387, 2886]
norms = []
for idx in sample_indices:
    vec  = np.array(dense_combined[idx]['embedding'], dtype='float32')
    faiss.normalize_L2(vec.reshape(1, -1))
    norms.append(abs(np.linalg.norm(vec) - 1.0) < 1e-5)
check(all(norms), f"Embeddings normalize to unit vectors")

# Content integrity
check(
    dense_combined[0]['metadata'].get('action') ==
    dense_actions[0]['metadata'].get('action'),
    f"First combined chunk matches first action chunk"
)
check(
    dense_combined[EXPECTED_ACTION_CHUNKS]['metadata'].get('policy_name') ==
    dense_policies[0]['metadata'].get('policy_name'),
    f"First policy in combined matches first policy chunk"
)

print(f"  → Dense embedding dim: {dense_index.d}")
print(f"  → Total vectors in FAISS: {dense_index.ntotal}")


# ══════════════════════════════════════════════════════════════════════
# SECTION 2: SPARSE EMBEDDING (SPLADE)
# ══════════════════════════════════════════════════════════════════════
print()
print("── 2. Sparse Embedding (SPLADE) ─────────────────────────────")

with open(os.path.join(OUTPUT_DIR, 'sparse', 'action_chunks_splade.json')) as f:
    splade_actions = json.load(f)
with open(os.path.join(OUTPUT_DIR, 'sparse', 'policy_chunks_splade.json')) as f:
    splade_policies = json.load(f)
with open(os.path.join(OUTPUT_DIR, 'sparse', 'combined_chunks_splade.json')) as f:
    splade_combined = json.load(f)

# Count checks
check(len(splade_actions)  == EXPECTED_ACTION_CHUNKS,
      f"Action chunks count",
      f"expected={EXPECTED_ACTION_CHUNKS} actual={len(splade_actions)}")
check(len(splade_policies) == EXPECTED_POLICY_CHUNKS,
      f"Policy chunks count",
      f"expected={EXPECTED_POLICY_CHUNKS} actual={len(splade_policies)}")
check(len(splade_combined) == EXPECTED_TOTAL_CHUNKS,
      f"Combined chunks count",
      f"expected={EXPECTED_TOTAL_CHUNKS} actual={len(splade_combined)}")

# Sparse embedding presence
missing_action = sum(
    1 for c in splade_actions
    if not c.get('sparse_embedding')
)
missing_policy = sum(
    1 for c in splade_policies
    if not c.get('sparse_embedding')
)
check(missing_action == 0, f"No missing action sparse embeddings",
      f"missing={missing_action}")
check(missing_policy == 0, f"No missing policy sparse embeddings",
      f"missing={missing_policy}")

# Sparse embedding format — should be dict with string keys
sample_sparse = splade_actions[0]['sparse_embedding']
check(isinstance(sample_sparse, dict),
      f"Sparse embedding is dictionary format")
check(all(isinstance(k, str) for k in list(sample_sparse.keys())[:5]),
      f"Sparse embedding keys are strings (token IDs)")
check(all(isinstance(v, float) for v in list(sample_sparse.values())[:5]),
      f"Sparse embedding values are floats (weights)")

# Token IDs should be valid BERT vocab indices
max_token_id = max(int(k) for k in sample_sparse.keys())
check(max_token_id < EXPECTED_VOCAB_SIZE,
      f"Token IDs within BERT vocab range",
      f"max_token_id={max_token_id} vocab_size={EXPECTED_VOCAB_SIZE}")

# Sparse vector size stats
action_sizes = [len(c['sparse_embedding']) for c in splade_actions]
policy_sizes = [len(c['sparse_embedding']) for c in splade_policies]
check(min(action_sizes) > 0,  f"No empty action sparse vectors")
check(min(policy_sizes) > 0,  f"No empty policy sparse vectors")

print(f"  → Action sparse avg tokens: {sum(action_sizes)/len(action_sizes):.0f}")
print(f"  → Policy sparse avg tokens: {sum(policy_sizes)/len(policy_sizes):.0f}")

# Order check on combined
splade_type_counts = Counter(
    c['metadata']['type'] for c in splade_combined
)
check(splade_type_counts['iam_action'] == EXPECTED_ACTION_CHUNKS,
      f"Combined action type count",
      f"expected={EXPECTED_ACTION_CHUNKS} "
      f"actual={splade_type_counts['iam_action']}")
check(splade_type_counts['iam_policy'] == EXPECTED_POLICY_CHUNKS,
      f"Combined policy type count",
      f"expected={EXPECTED_POLICY_CHUNKS} "
      f"actual={splade_type_counts['iam_policy']}")


# ══════════════════════════════════════════════════════════════════════
# SECTION 3: BM25
# ══════════════════════════════════════════════════════════════════════
print()
print("── 3. BM25 ──────────────────────────────────────────────────")

with open(os.path.join(OUTPUT_DIR, 'bm25', 'action_bm25.pkl'), 'rb') as f:
    bm25_action_index = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'bm25', 'policy_bm25.pkl'), 'rb') as f:
    bm25_policy_index = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'bm25', 'combined_bm25.pkl'), 'rb') as f:
    bm25_combined_index = pickle.load(f)

with open(os.path.join(OUTPUT_DIR, 'bm25', 'action_chunks_mapped.json')) as f:
    bm25_action_chunks = json.load(f)
with open(os.path.join(OUTPUT_DIR, 'bm25', 'policy_chunks_mapped.json')) as f:
    bm25_policy_chunks = json.load(f)
with open(os.path.join(OUTPUT_DIR, 'bm25', 'combined_chunks_mapped.json')) as f:
    bm25_combined_chunks = json.load(f)

# Count checks
check(bm25_action_index.corpus_size == EXPECTED_ACTION_CHUNKS,
      f"BM25 action index corpus size",
      f"expected={EXPECTED_ACTION_CHUNKS} "
      f"actual={bm25_action_index.corpus_size}")
check(bm25_policy_index.corpus_size == EXPECTED_POLICY_CHUNKS,
      f"BM25 policy index corpus size",
      f"expected={EXPECTED_POLICY_CHUNKS} "
      f"actual={bm25_policy_index.corpus_size}")
check(bm25_combined_index.corpus_size == EXPECTED_TOTAL_CHUNKS,
      f"BM25 combined index corpus size",
      f"expected={EXPECTED_TOTAL_CHUNKS} "
      f"actual={bm25_combined_index.corpus_size}")

# Chunk mapping counts
check(len(bm25_action_chunks)   == EXPECTED_ACTION_CHUNKS,
      f"BM25 action chunks mapped count",
      f"expected={EXPECTED_ACTION_CHUNKS} "
      f"actual={len(bm25_action_chunks)}")
check(len(bm25_policy_chunks)   == EXPECTED_POLICY_CHUNKS,
      f"BM25 policy chunks mapped count",
      f"expected={EXPECTED_POLICY_CHUNKS} "
      f"actual={len(bm25_policy_chunks)}")
check(len(bm25_combined_chunks) == EXPECTED_TOTAL_CHUNKS,
      f"BM25 combined chunks mapped count",
      f"expected={EXPECTED_TOTAL_CHUNKS} "
      f"actual={len(bm25_combined_chunks)}")

# Index and chunks are in sync
check(
    bm25_combined_index.corpus_size == len(bm25_combined_chunks),
    f"BM25 combined index size matches chunks list"
)

# BM25 internal statistics present
check(hasattr(bm25_combined_index, 'idf'),
      f"BM25 index has IDF weights")
check(hasattr(bm25_combined_index, 'doc_len'),
      f"BM25 index has document lengths")
check(hasattr(bm25_combined_index, 'avgdl'),
      f"BM25 index has average document length")

# Quick retrieval test
import re
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s:\-]', ' ', text)
    return text.split()

test_scores = bm25_combined_index.get_scores(
    tokenize("allow lambda to read objects from s3")
)
check(len(test_scores) == EXPECTED_TOTAL_CHUNKS,
      f"BM25 scores array length matches corpus",
      f"expected={EXPECTED_TOTAL_CHUNKS} actual={len(test_scores)}")
check(max(test_scores) > 0,
      f"BM25 returns non-zero scores for test query",
      f"max_score={max(test_scores):.4f}")

top5_indices = np.argsort(test_scores)[::-1][:5]
print(f"  → BM25 top 5 for 'allow lambda to read objects from s3':")
for idx in top5_indices:
    chunk    = bm25_combined_chunks[idx]
    metadata = chunk['metadata']
    identifier = (metadata.get('action') or
                  metadata.get('policy_name', 'unknown'))
    print(f"    [{metadata['type']:<10}] {identifier:<50} "
          f"score={test_scores[idx]:.4f}")


# ══════════════════════════════════════════════════════════════════════
# SECTION 4: CROSS-METHOD CONSISTENCY
# ══════════════════════════════════════════════════════════════════════
print()
print("── 4. Cross-Method Consistency ──────────────────────────────")

# All three methods should have same chunk texts in same order
check(
    dense_combined[0]['text'] == splade_combined[0]['text'],
    f"Dense and SPLADE combined have same first chunk text"
)
check(
    dense_combined[0]['text'] == bm25_combined_chunks[0]['text'],
    f"Dense and BM25 combined have same first chunk text"
)
check(
    dense_combined[-1]['text'] == splade_combined[-1]['text'],
    f"Dense and SPLADE combined have same last chunk text"
)
check(
    dense_combined[-1]['text'] == bm25_combined_chunks[-1]['text'],
    f"Dense and BM25 combined have same last chunk text"
)

# Same metadata at same positions
check(
    dense_combined[100]['metadata']['action'] ==
    splade_combined[100]['metadata']['action'],
    f"Dense and SPLADE have same action at position 100"
)
check(
    dense_combined[EXPECTED_ACTION_CHUNKS]['metadata']['policy_name'] ==
    splade_combined[EXPECTED_ACTION_CHUNKS]['metadata']['policy_name'],
    f"Dense and SPLADE have same policy at boundary position"
)


# ══════════════════════════════════════════════════════════════════════
# FINAL VERDICT
# ══════════════════════════════════════════════════════════════════════
print()
print("=" * 60)
print(f"OVERALL RESULT: "
      f"{'✓ ALL CHECKS PASSED' if all_checks_passed else '✗ SOME CHECKS FAILED'}")
print("=" * 60)
print()
print("Summary:")
print(f"  Action chunks:    {EXPECTED_ACTION_CHUNKS}")
print(f"  Policy chunks:    {EXPECTED_POLICY_CHUNKS}")
print(f"  Total chunks:     {EXPECTED_TOTAL_CHUNKS}")
print()
print(f"  Dense FAISS dim:  {dense_index.d}")
print(f"  Dense vectors:    {dense_index.ntotal}")
print()
print(f"  SPLADE action avg sparse tokens: "
      f"{sum(action_sizes)/len(action_sizes):.0f}")
print(f"  SPLADE policy avg sparse tokens: "
      f"{sum(policy_sizes)/len(policy_sizes):.0f}")
print()
print(f"  BM25 combined corpus size: {bm25_combined_index.corpus_size}")
print(f"  BM25 avg doc length:       {bm25_combined_index.avgdl:.1f} tokens")

COMPLETE EMBEDDING VERIFICATION

── 1. Dense Embedding (Voyage) ──────────────────────────────
  ✓ Action chunks count
    expected=4505 actual=4505
  ✓ Policy chunks count
    expected=500 actual=500
  ✓ Combined chunks count
    expected=5005 actual=5005
  ✓ FAISS index count
    expected=5005 actual=5005
  ✓ FAISS dimension
    expected=1024 actual=1024
  ✓ Action embedding dimension
    expected=1024 actual=1024
  ✓ No missing action embeddings
    missing=0
  ✓ No missing policy embeddings
    missing=0
  ✓ Combined action type count
    expected=4505 actual=4505
  ✓ Combined policy type count
    expected=500 actual=500
  ✓ Combined starts with action
  ✓ Combined ends with policy
  ✓ Embeddings normalize to unit vectors
  ✓ First combined chunk matches first action chunk
  ✓ First policy in combined matches first policy chunk
  → Dense embedding dim: 1024
  → Total vectors in FAISS: 5005

── 2. Sparse Embedding (SPLADE) ─────────────────────────────
  ✓ Action chunks count
    e